# Import necessary liberaies & modules

In [ ]:
from ConnectFourEnv2side import ConnectFourEnv
from Players.ChildSmarterPlayer import ChildSmarterPlayer
from Players.BabySmarterPlayer import BabySmarterPlayer
from Players.ModelPlayer import ModelPlayer
from Players.TeenagerSmarterPlayer import TeenagerSmarterPlayer
from Players.BabyPlayer import BabyPlayer
from Players.ChildPlayer import ChildPlayer
from Players.TeenagerPlayer import TeenagerPlayer
from Players.AdultPlayer import AdultPlayer
from Players.AdultSmarterPlayer import AdultSmarterPlayer
from stable_baselines3 import PPO
from stable_baselines3 import DQN
import os
import random
from EloLeaderboard import EloLeaderboard
import matplotlib.pyplot as plt
from stable_baselines3.common.torch_layers import NatureCNN
from stable_baselines3.common.env_checker import check_env
import random
from CustomCnn import CustomCNN

# Baseline

In [ ]:
env = ConnectFourEnv(opponent=BabyPlayer())
model = PPO("MlpPolicy", env, verbose=1,)
model.learn(total_timesteps=(100000))

# PPO

In [ ]:
env = ConnectFourEnv()

model = PPO("MlpPolicy", env, verbose=1, learning_rate=2.5e-4,ent_coef=0.01,gamma=0.99)

opponents = [BabyPlayer(), BabySmarterPlayer(), ChildPlayer(), ChildSmarterPlayer(), TeenagerPlayer(), TeenagerSmarterPlayer(), AdultPlayer(), AdultSmarterPlayer()]

Elos = []
Models = []
Timesteps = [100000,120000,200000,250000,300000,400000,600000,800000]

for i in range(8):
    env.change_opponent(opponents[i])
    model.set_env(env)
    model.learn(total_timesteps=Timesteps[i])
    myModelPlayer = ModelPlayer(model,name="Your trained Model1")
    Elos.append(EloLeaderboard().get_elo(myModelPlayer, num_matches=200))
    Models.append(model)

print(Elos)

In [ ]:
plt.figure()
plt.plot(Elos)
plt.title('PPO & Mlp')
plt.ylabel('Elo Score')

# DQN

In [ ]:
env = ConnectFourEnv()

model = DQN("MlpPolicy", env, verbose=1, learning_rate=1e-4, gamma=0.99, buffer_size=100000)

opponents = [BabyPlayer(), BabySmarterPlayer(), ChildPlayer(), ChildSmarterPlayer(), TeenagerPlayer(), TeenagerSmarterPlayer(), AdultPlayer(), AdultSmarterPlayer()]

Elos = []
Models = []
Timesteps = [100000,120000,200000,250000,300000,350000,400000,600000]

for i in range(8):
    env.change_opponent(opponents[i])
    model.set_env(env)
    model.learn(total_timesteps=Timesteps[i])
    myModelPlayer = ModelPlayer(model,name="Your trained Model1")
    Elos.append(EloLeaderboard().get_elo(myModelPlayer, num_matches=200))
    Models.append(model)

print(Elos)

In [ ]:
plt.figure()
plt.plot(Elos)
plt.title('Self-play-training')
plt.ylabel('Elo Score')

# Self-Play_Training

In [ ]:
env = ConnectFourEnv()
model = DQN.load("Models/best.zip")

Elos = []

for i in range(5):
    opponent = ModelPlayer(model,name="yourself")
    env.change_opponent(opponent)
    model.set_env(env)
    model.learn(total_timesteps=50000)
    myModelPlayer = ModelPlayer(model,name="Your trained Model1")
    Elos.append(EloLeaderboard().get_elo(myModelPlayer, num_matches=200))

# Save model & Load for test(optional)

In [ ]:
randNum = random.randint(0, 10000)
randNum = str(randNum)
model.save(f'Models/temp_model_{randNum}')

In [ ]:
#visualize
model = DQN.load("Models/DQN1870.zip")
env = ConnectFourEnv(opponent= AdultSmarterPlayer(), render_mode="human")

obs , _=  env.reset()
for i in range(100):
    action, _states = model.predict(obs,deterministic=True)
    obs, rewards, dones, truncated,info = env.step(action)
    env.render()
    if(truncated or dones):
        obs , _=  env.reset()

# Get Elo

In [ ]:
model1 = DQN.load("Models/DQN1870.zip")
myModelPlayer1 = ModelPlayer(model1,name="Your trained Model1")

print(EloLeaderboard().get_elo(myModelPlayer1, num_matches=200))